In [1]:
import pandas as pd
import numpy as np
import time
from openpyxl import load_workbook

# Read input data
data = pd.read_excel('data/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

penalty_factor = 1e6

# Read power demand data
power_demand_data = pd.read_excel('data/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# Initialize an empty list to store results for each demand
all_outputs = []

# Define function to calculate power produced
def calculate_power(P):
    return np.sum(P)

# Define function to calculate fuel cost with penalty
def calculate_fuel_cost_with_penalty(P, a, b, c, demand):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.multiply(b, P) + np.multiply(c, squaredP)), 1000)

    # Penalty for power balance constraint violation
    power_difference = np.sum(P) - demand
    penalty_term = penalty_factor * abs(power_difference)

    total_Fuel_Cost = np.sum(Fuel_Cost) + penalty_term
    return total_Fuel_Cost

# Interval Halving method
def interval_halving_search(func, a, b, tol=1e-6):
    while (b - a) > tol:
        midpoint = (a + b) / 2
        x1 = (a + midpoint) / 2
        x2 = (b + midpoint) / 2
        if func(x1) < func(midpoint):
            b = midpoint
        elif func(x2) < func(midpoint):
            a = midpoint
        else:
            a = x1
            b = x2
    return (a + b) / 2

def hybrid_search(func, a, b, tol=1e-6):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return interval_halving_search(func, a, b, tol)
    else:
        results = []
        for ai, bi in zip(a, b):
            results.append(interval_halving_search(func, ai, bi, tol))
        return np.mean(results)  # Or return the results as needed

# Optimization loop
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    best_position = None
    best_cost = np.inf

    for i in range(len(Minimum_Capacity)):
        P = hybrid_search(lambda x: calculate_fuel_cost_with_penalty(x * np.ones_like(Minimum_Capacity), a, b, c, demand), Minimum_Capacity[i], Maximum_Capacity[i])
        
        # Aggressive enforcement of capacity constraints
        for _ in range(10):  # You can adjust the number of iterations as needed
            P = np.minimum(np.maximum(P, Minimum_Capacity), Maximum_Capacity)
            
            # Adjust for power balance constraint
            power_difference = np.sum(P) - demand
            if power_difference > 0:
                # Reduce overproduced power
                for j in range(len(P)):
                    if P[j] > Minimum_Capacity[j]:
                        reduction = min(P[j] - Minimum_Capacity[j], power_difference)
                        P[j] -= reduction
                        power_difference -= reduction
            elif power_difference < 0:
                # Increase underproduced power
                for j in range(len(P)):
                    if P[j] < Maximum_Capacity[j]:
                        increase = min(Maximum_Capacity[j] - P[j], abs(power_difference))
                        P[j] += increase
                        power_difference += increase
            
            cost = calculate_fuel_cost_with_penalty(P * np.ones_like(Minimum_Capacity), a, b, c, demand)
            if cost < best_cost:
                best_cost = cost
                best_position = P.copy()
            else:
                break  # Exit the loop if cost increases
            
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': best_position})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(best_position):.2f} MW")
    print(f'Total Fuel Cost: Rp {best_cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(best_position),
                        'Total Fuel Cost': best_cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = best_position[unit_index]
        output_info[f'Unit {unit} Cost'] = best_cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = 'IH.xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'IH.xlsx'")

Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           250.000000
1      2           250.000000
2      3           250.000000
3      4           250.000000
4      5           408.000000
5      6           408.000000
6      7           408.000000
7      8           142.424002
8      9           170.000000
9     10           170.000000
10    11           170.000000
11    12            80.000000
12    13            80.000000
13    14           170.000000
14    15           170.000000
15    16           220.000000
16    17           220.000000
17    18           220.000000
18    19            16.000000
19    20            16.000000
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31           